# scikit-learn

In [1]:
import pandas as pd

df = pd.read_csv('../data/ds_salaries.csv')
df

,work_year,experience_level,employment_type,job_title,salary,salary_currency,salary_in_usd,employee_residence,remote_ratio,company_location,company_size
0,2023,SE,FT,Principal Data Scientist,80000,EUR,85847,ES,100,ES,L
1,2023,MI,CT,ML Engineer,30000,USD,30000,US,100,US,S
2,2023,MI,CT,ML Engineer,25500,USD,25500,US,100,US,S
3,2023,SE,FT,Data Scientist,175000,USD,175000,CA,100,CA,M
4,2023,SE,FT,Data Scientist,120000,USD,120000,CA,100,CA,M
...,...,...,...,...,...,...,...,...,...,...,...
3750,2020,SE,FT,Data Scientist,412000,USD,412000,US,100,US,L
3751,2021,MI,FT,Principal Data Scientist,151000,USD,151000,US,100,US,L
3752,2020,EN,FT,Data Scientist,105000,USD,105000,US,100,US,S
3753,2020,EN,CT,Business Data Analyst,100000,USD,100000,US,100,US,L


## データ前処理

In [2]:
# 不要列削除
df = df.drop(columns=["salary", 'salary_currency'])

In [3]:
# 列の並び替え
df = df[['work_year', 'experience_level', 'employment_type', 
         'job_title', 'employee_residence', 'company_location', 
         'company_size','remote_ratio','salary_in_usd']]

In [4]:
# 説明変数と目的変数の分離
X = df.drop("salary_in_usd", axis=1)
y = df["salary_in_usd"]

In [30]:
# train_test分割
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=123)

In [31]:
# 少数カテゴリをOtherに変換
import numpy as np

target_columns = ['job_title', 'employee_residence', 'company_location']
for target_column in target_columns:
    counts = X_train[target_column].value_counts()
    majors = counts[counts >= 30].index
    
    X_train[target_column + '_mod'] = np.where(
        X_train[target_column].isin(majors), 
        X_train[target_column], 
        'Other', 
    )
    
    X_test[target_column + '_mod'] = np.where(
        X_test[target_column].isin(majors), 
        X_test[target_column], 
        'Other', 
    )

# 不要列削除
X_train = X_train.drop(columns=['job_title', 'employee_residence', 'company_location'])
X_test = X_test.drop(columns=['job_title', 'employee_residence', 'company_location'])

In [32]:
# ダミー変数化
X_train = pd.get_dummies(X_train, columns=['experience_level', 'employment_type', 'company_size', 
                                           'job_title_mod', 'employee_residence_mod',
                                           'company_location_mod'], dtype=int)
X_test = pd.get_dummies(X_test, columns=['experience_level', 'employment_type', 'company_size', 
                                         'job_title_mod', 'employee_residence_mod',
                                         'company_location_mod'], dtype=int)

# 列を揃える
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

## バリデーション設計

### クロスバリデーション

In [8]:
# ML用
from sklearn.model_selection import KFold

n_splits = 5
cv = list(KFold(n_splits=n_splits, shuffle=True, random_state=123).split(X_train))

for nfold, (train_index, valid_index) in enumerate(cv):
    print('-'*20, nfold, '-'*20)
    X_train_fold = X_train.iloc[train_index]
    y_train_fold = y_train.iloc[train_index]
    
    X_valid_fold = X_train.iloc[valid_index]
    y_valid_fold = y_train.iloc[valid_index]
    
    print(X_train_fold.shape, y_train_fold.shape)
    print(X_valid_fold.shape, y_valid_fold.shape)
    print(f'y_train : {y_train.mean()}, y_train_fold : {y_train_fold.mean()}, y_valid_fold : {y_valid_fold.mean()}')

-------------------- 0 --------------------
(2403, 37) (2403,)
(601, 37) (601,)
y_train : 138580.20539280958, y_train_fold : 138685.56346233873, y_valid_fold : 138158.94841930116
-------------------- 1 --------------------
(2403, 37) (2403,)
(601, 37) (601,)
y_train : 138580.20539280958, y_train_fold : 138513.01872659175, y_valid_fold : 138848.84026622295
-------------------- 2 --------------------
(2403, 37) (2403,)
(601, 37) (601,)
y_train : 138580.20539280958, y_train_fold : 137816.09696213066, y_valid_fold : 141635.3677204659
-------------------- 3 --------------------
(2403, 37) (2403,)
(601, 37) (601,)
y_train : 138580.20539280958, y_train_fold : 139056.01872659175, y_valid_fold : 136677.74376039935
-------------------- 4 --------------------
(2404, 37) (2404,)
(600, 37) (600,)
y_train : 138580.20539280958, y_train_fold : 138830.22504159732, y_valid_fold : 137578.46


## モデル学習（LinearRegression）

In [21]:
import numpy as np
from sklearn.model_selection import KFold
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.metrics import mean_squared_error

n_splits = 5
cv = list(KFold(n_splits=n_splits, shuffle=True, random_state=123).split(X_train))
scores = []

for nfold, (train_index, valid_index) in enumerate(cv):
    print('-'*20, nfold, '-'*20)
    X_train_fold = X_train.iloc[train_index]
    y_train_fold = y_train.iloc[train_index]
    
    X_valid_fold = X_train.iloc[valid_index]
    y_valid_fold = y_train.iloc[valid_index]


    model_lr = LinearRegression()
    model_lr.fit(X_train_fold, y_train_fold)

    pred = model_lr.predict(X_valid_fold)

    r2 = r2_score(y_valid_fold, pred)
    rmse = np.sqrt(mean_squared_error(y_valid_fold, pred))
    print('R2 :', r2)
    print('RMSE :', rmse)

    scores.append({'fold': nfold, 'R2': r2, 'RMSE': rmse})

print('-'*18, 'total', '-'*18)
score_df = pd.DataFrame(scores)
print(score_df)
print('\nCV mean')
print(score_df[['R2', 'RMSE']].mean())

-------------------- 0 --------------------
R2 : 0.36200365801435497
RMSE : 50621.572413207716
-------------------- 1 --------------------
R2 : 0.40531619374044103
RMSE : 48874.2228635808
-------------------- 2 --------------------
R2 : 0.36116540626934523
RMSE : 53557.79912942413
-------------------- 3 --------------------
R2 : 0.4000795148212549
RMSE : 46134.282566350485
-------------------- 4 --------------------
R2 : 0.4015922837239675
RMSE : 50213.475093089204
------------------ total ------------------
   fold        R2          RMSE
0     0  0.362004  50621.572413
1     1  0.405316  48874.222864
2     2  0.361165  53557.799129
3     3  0.400080  46134.282566
4     4  0.401592  50213.475093

CV mean
R2          0.386031
RMSE    49880.270413
dtype: float64


## モデル学習（LinearRegression、最終）

In [22]:
from sklearn.linear_model import LinearRegression

final_model_lr = LinearRegression()
final_model_lr.fit(X_train, y_train)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for the `lsqr` solver.`tol` is set as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. This parameter has no effect when fittingon dense data... versionadded:: 1.7",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False


## モデル推論（LinearRegression）

In [23]:
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score
import numpy as np

pred = final_model_lr.predict(X_test)

r2 = r2_score(y_test, pred)
rmse = np.sqrt(mean_squared_error(y_test, pred))

print('R2 :', r2)
print('RMSE :', rmse)

R2 : 0.44274578683052357
RMSE : 44885.897753106015


In [24]:
coef = pd.Series(final_model.coef_, index=X_train.columns)
coef.sort_values(ascending=False).head(10)

experience_level_EX                        52697.931842
employee_residence_mod_US                  32303.447869
job_title_mod_Data Science Manager         26796.312956
job_title_mod_Research Scientist           26147.094773
job_title_mod_Applied Scientist            21856.287338
company_location_mod_US                    20743.596193
employee_residence_mod_CA                  19339.633796
company_location_mod_CA                    12108.595547
employment_type_FT                         11539.473999
job_title_mod_Machine Learning Engineer     9294.795748
dtype: float64

In [25]:
coef = pd.Series(final_model.coef_, index=X_train.columns)
coef.sort_values().head(10)

job_title_mod_Data Analyst         -41281.071452
experience_level_EN                -38227.107678
employee_residence_mod_IN          -37548.877004
company_location_mod_ES            -31727.710647
experience_level_MI                -20099.931441
employee_residence_mod_Other       -17873.174205
job_title_mod_Data Engineer        -13518.325336
company_location_mod_IN            -10015.631143
job_title_mod_Analytics Engineer    -9919.681620
company_size_S                      -8773.215883
dtype: float64

## モデル学習（RandomForestRegressor）

In [17]:
import numpy as np
from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score
from sklearn.metrics import mean_squared_error

n_splits = 5
cv = list(KFold(n_splits=n_splits, shuffle=True, random_state=123).split(X_train))
scores = []

for nfold, (train_index, valid_index) in enumerate(cv):
    print('-'*20, nfold, '-'*20)
    X_train_fold = X_train.iloc[train_index]
    y_train_fold = y_train.iloc[train_index]
    
    X_valid_fold = X_train.iloc[valid_index]
    y_valid_fold = y_train.iloc[valid_index]

    model_rf = RandomForestRegressor(max_depth=5, random_state=0)
    model_rf.fit(X_train_fold, y_train_fold)

    pred = model_rf.predict(X_valid_fold)

    r2 = r2_score(y_valid_fold, pred)
    rmse = np.sqrt(mean_squared_error(y_valid_fold, pred))
    print('R2 :', r2)
    print('RMSE :', rmse)

    scores.append({'fold': nfold, 'R2': r2, 'RMSE': rmse})

print('-'*18, 'total', '-'*18)
score_df = pd.DataFrame(scores)
print(score_df)
print('\nCV mean')
print(score_df[['R2', 'RMSE']].mean())

-------------------- 0 --------------------
R2 : 0.3383361400685956
RMSE : 51551.96737293479
-------------------- 1 --------------------
R2 : 0.41290215206473124
RMSE : 48561.49550383366
-------------------- 2 --------------------
R2 : 0.34955322906007014
RMSE : 54042.37046188781
-------------------- 3 --------------------
R2 : 0.3784543958775254
RMSE : 46958.41450044117
-------------------- 4 --------------------
R2 : 0.4065762308429761
RMSE : 50003.93186218935
------------------ total ------------------
   fold        R2          RMSE
0     0  0.338336  51551.967373
1     1  0.412902  48561.495504
2     2  0.349553  54042.370462
3     3  0.378454  46958.414500
4     4  0.406576  50003.931862

CV mean
R2          0.377164
RMSE    50223.635940
dtype: float64


## モデル学習（RandomForestRegressor、最終）

In [18]:
from sklearn.ensemble import RandomForestRegressor

final_model_rf = RandomForestRegressor(max_depth=5, random_state=0)
final_model_rf.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""squared_error"", ""absolute_error"", ""friedman_mse"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""friedman_mse"", which usesmean squared error with Friedman's improvement score for potentialsplits, ""absolute_error"" for the mean absolute error, which minimizesthe L1 loss using the median of each terminal node, and ""poisson"" whichuses reduction in Poisson deviance to find splits.Training using ""absolute_error"" is significantly slowerthan when using ""squared_error""... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",5
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples a

## モデル推論（RandomForestRegressor）

In [19]:
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score
import numpy as np

pred = final_model_rf.predict(X_test)

r2 = r2_score(y_test, pred)
rmse = np.sqrt(mean_squared_error(y_test, pred))

print('R2 :', r2)
print('RMSE :', rmse)

R2 : 0.4310760511710052
RMSE : 45353.451492590175


In [20]:
importance = pd.Series(model_rf.feature_importances_, index=X_train.columns)
importance.sort_values(ascending=False).head(10)

employee_residence_mod_US     0.546377
job_title_mod_Data Analyst    0.131960
experience_level_SE           0.068776
experience_level_EX           0.038285
experience_level_EN           0.032508
company_location_mod_CA       0.020407
employee_residence_mod_CA     0.017696
work_year                     0.015473
experience_level_MI           0.014018
remote_ratio                  0.013649
dtype: float64